In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("--- Step 1: Fetching Data Directly via URL ---")

# Pulling the genuine Pima Indians Diabetes dataset directly from a verified public repository
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

# The raw file doesn't have headers, so we define the exact column names from the Kaggle dataset
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

try:
    df = pd.read_csv(url, names=column_names)
    print("✅ Success! Dataset loaded directly from the web link. No local file needed.\n")
except Exception as e:
    print(f"❌ Connection failed: {e}")

# Handle missing values encoded as 0
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

# Separate features and target
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# Split and Scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------------------
# STEP 2: BASELINE MODEL CREATION
# ---------------------------------------------------------------------
print("--- Step 2: Training Baseline Model ---")
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

y_pred_base = baseline_model.predict(X_test_scaled)
y_prob_base = baseline_model.predict_proba(X_test_scaled)[:, 1]
print("✅ Baseline model trained successfully.\n")

# ---------------------------------------------------------------------
# STEP 3: HYPERPARAMETER TUNING
# ---------------------------------------------------------------------
print("--- Step 3: Hyperparameter Tuning (Running Grid Search...) ---")
# Structured parameter grid for quick, efficient tuning
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)
print(f"🎯 Best Parameters Found: {grid_search.best_params_}\n")

optimized_model = grid_search.best_estimator_

# ---------------------------------------------------------------------
# STEP 4: CROSS-VALIDATION IMPLEMENTATION
# ---------------------------------------------------------------------
print("--- Step 4: Cross-Validation Verification ---")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(optimized_model, X_train_scaled, y_train, cv=kf, scoring='accuracy')
print(f"📊 Mean 5-Fold CV Accuracy: {cv_scores.mean():.4f}\n")

# ---------------------------------------------------------------------
# STEP 5: MODEL COMPARISON & FINAL EVALUATION
# ---------------------------------------------------------------------
print("--- Step 5: Final Performance Comparison ---")
y_pred_opt = optimized_model.predict(X_test_scaled)
y_prob_opt = optimized_model.predict_proba(X_test_scaled)[:, 1]

def get_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

comparison_df = pd.DataFrame({
    'Baseline Model': get_metrics(y_test, y_pred_base, y_prob_base),
    'Optimized Model': get_metrics(y_test, y_pred_opt, y_prob_opt)
})
print(comparison_df.round(4))

--- Step 1: Fetching Data Directly via URL ---
✅ Success! Dataset loaded directly from the web link. No local file needed.

--- Step 2: Training Baseline Model ---
✅ Baseline model trained successfully.

--- Step 3: Hyperparameter Tuning (Running Grid Search...) ---
🎯 Best Parameters Found: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}

--- Step 4: Cross-Validation Verification ---
📊 Mean 5-Fold CV Accuracy: 0.7622

--- Step 5: Final Performance Comparison ---
           Baseline Model  Optimized Model
Accuracy           0.7792           0.7532
Precision          0.7174           0.6667
Recall             0.6111           0.5926
F1-Score           0.6600           0.6275
ROC-AUC            0.8179           0.8085
